# Ames Real Estate Machine Learning & Explainable AI Pipeline

**Dataset**: Ames Housing Dataset (2,930 transactions, 80 features)  
**Objective**: Build a production-grade, domain-constrained regression pipeline using **Monotonic XGBoost** to accurately forecast single-family property valuation with Explainable AI (XAI) feature interpretability.

---

## 1. Environment Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from pathlib import Path
from sklearn.model_selection import KFold, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

dataset_path = Path('../data/AmesHousing.csv')
df_raw = pd.read_csv(dataset_path)
print(f"Raw Dataset Loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns.")
df_raw[['Overall Qual', 'Gr Liv Area', 'Garage Cars', 'Full Bath', 'Bedroom AbvGr', 'Year Built', 'SalePrice']].head()

## 2. Exploratory Data Analysis & Outlier Detection

In accordance with standard Ames Housing dataset documentation (De Cock, 2011), properties with living area (`Gr Liv Area`) exceeding **4,000 sq ft** represent partial sales or extreme statistical outliers that distort regression models. We filter these anomalies along with missing target entries.

In [ ]:
base_features = ['Overall Qual', 'Gr Liv Area', 'Garage Cars', 'Full Bath', 'Bedroom AbvGr', 'Year Built']
all_needed_cols = base_features + ['SalePrice']

# Remove > 4000 sq ft outliers & handle null entries
df_clean = df_raw[(df_raw['Gr Liv Area'] < 4000) & (df_raw['SalePrice'] > 0)].dropna(subset=all_needed_cols).copy()
print(f"Initial Record Count: {len(df_raw)}")
print(f"Cleaned Record Count: {len(df_clean)} (Removed {len(df_raw) - len(df_clean)} anomalies)")

print("\nTarget Variable ('SalePrice') Summary Statistics:")
print(df_clean['SalePrice'].describe().apply(lambda x: f"${x:,.2f}"))

## 3. Feature Engineering Strategy

We introduce a domain-driven feature: **Quality-to-Area Interaction** (`Overall Qual` * `Gr Liv Area`). This captures non-linear price amplification where additional living area adds significantly more monetary value in high-quality luxury homes than in lower-grade structures.

In [ ]:
X = df_clean[base_features].copy()
qual_map = {'Very_Poor': 1, 'Poor': 2, 'Fair': 3, 'Below_Average': 4, 'Average': 5, 'Above_Average': 6, 'Good': 7, 'Very_Good': 8, 'Excellent': 9, 'Very_Excellent': 10}
if X['Overall Qual'].dtype == object:
    X['Overall Qual'] = X['Overall Qual'].map(qual_map).fillna(5)
X['Overall Qual'] = pd.to_numeric(X['Overall Qual'], errors='coerce').fillna(5).astype(float)
X['Qual_Area_Interaction'] = X['Overall Qual'] * X['Gr Liv Area']
y = df_clean['SalePrice']

print(f"Feature Matrix Shape: {X.shape}")
X.head()

## 4. Model Benchmarks (5-Fold Cross-Validation)

We evaluate three regression paradigms across 5 stratified folds:
1. **Linear Regression** (Baseline)
2. **Random Forest Regressor** (Ensemble Baseline)
3. **Monotonic XGBoost Regressor** (Domain-Constrained Gradient Boosting)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Linear Regression (Baseline)": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42),
    "Monotonic XGBoost Regressor": xgb.XGBRegressor(
        n_estimators=220,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.85,
        colsample_bytree=0.85,
        monotone_constraints='(1, 1, 1, 1, 0, 1, 1)',
        random_state=42
    )
}

results = []
for name, model in models.items():
    cv_res = cross_validate(
        model, X, y, cv=kf,
        scoring={'r2': 'r2', 'rmse': 'neg_root_mean_squared_error', 'mae': 'neg_mean_absolute_error'}
    )
    mean_r2 = cv_res['test_r2'].mean()
    mean_rmse = -cv_res['test_rmse'].mean()
    mean_mae = -cv_res['test_mae'].mean()
    results.append({
        "Model": name,
        "R^2 Score": round(mean_r2, 4),
        "RMSE ($)": f"${mean_rmse:,.2f}",
        "MAE ($)": f"${mean_mae:,.2f}"
    })

pd.DataFrame(results)

## 5. Feature Importance & Explainable AI (XAI)

We inspect relative feature weight distributions (Gain) derived from the trained Monotonic XGBoost model.

In [ ]:
final_model = models["Monotonic XGBoost Regressor"]
final_model.fit(X, y)

importances = final_model.feature_importances_
feat_imp = pd.DataFrame({
    'Feature': X.columns,
    'Importance (%)': (importances * 100).round(2)
}).sort_values('Importance (%)', ascending=False)

feat_imp